# Explainable AI for Maternal Health Risk Stratification in Kenya
### Predicting Missed Timely Postnatal Care (PNC) — CRISP-DM Notebook

This notebook follows the CRISP-DM methodology end-to-end. Each phase is its own top-level section (`##`) so the whole team works in one file instead of separate notebooks.

**Phases:** 1. Business Understanding · 2. Data Understanding · 3. Data Preparation · 4. Modeling · 5. Evaluation · 6. Deployment

## **Explainable Machine Learning for Predicting Missed Timely Postnatal Care in Kenya**

## 1. Business Understanding

## 1.1 Business Overview

Maternal healthcare is a continuum that includes antenatal care (ANC), delivery care, and postnatal care (PNC). While Kenya has made considerable progress in maternal healthcare utilization, gaps remain in the continuity of care after childbirth.

According to the Kenya Demographic and Health Survey (KDHS) 2022, approximately 98% of women who had a live birth or stillbirth in the two years preceding the survey received ANC from a skilled provider, while about two-thirds attended at least four ANC visits. In addition, 88% of live births occurred in a health facility. However, only 78% of women with a live birth in the two years preceding the survey received a postnatal check within the first two days after birth.

These figures indicate that although contact with the health system is relatively high during pregnancy and delivery, timely postnatal follow-up is not universal. The gap is also uneven across population groups. For example, the KDHS 2022 reports that timely postnatal checks were received by 79% of women in urban areas compared with 69% in rural areas, while coverage ranged from 59% among women in the lowest wealth quintile to 83% among those in the highest wealth quintile.

The availability of detailed maternal-health data from the KDHS 2022 creates an opportunity to use data science to identify characteristics associated with missed timely postnatal care and support more targeted follow-up.

## 1.2  Problem Statement

Despite high utilization of antenatal and facility-based delivery services in Kenya, a proportion of women do not receive a postnatal health check within the first two days after childbirth. Current aggregate statistics describe the size of the gap but do not identify which individual women are more likely to miss timely postnatal care.

Healthcare workers responsible for community-level follow-up may therefore have limited information to prioritize women who could benefit most from additional follow-up.

This project seeks to address this gap by developing an explainable machine-learning model that predicts the likelihood of a woman missing timely postnatal care using demographic, socioeconomic, ANC, delivery, and healthcare-access characteristics from the KDHS 2022 dataset.

The model will not replace healthcare professionals or provide a clinical diagnosis. Instead, it will serve as a decision-support prototype that can help maternal and child health teams identify higher-risk profiles and understand the factors contributing to each prediction.

## 1.3 Business Objective

**Main Objective**

- To develop an explainable binary classification model that predicts the likelihood of women missing timely postnatal care in Kenya and provides understandable explanations of the factors contributing to each prediction.

**Specific Objectives**

- To identify and prepare relevant demographic, socioeconomic, ANC, delivery, and healthcare-access variables from the KDHS 2022 dataset.
- To examine the relationship between these factors and timely postnatal-care utilization.
- To develop a Logistic Regression model as a baseline for predicting missed timely postnatal care.
- To develop an XGBoost classification model and compare its predictive performance with the baseline model.
- To evaluate model performance using PR-AUC, recall, precision, F1-score, and ROC-AUC.
- To apply SHAP to explain the factors influencing individual predictions and overall model behaviour.
- To develop an interactive prototype that provides a predicted risk score and the main factors contributing to the prediction.
- To visualize predicted risk at DHS cluster level using the available GPS data as a presentation layer.

## 1.4 Stakeholders

**Stakeholder	Role / Interest**

- County Ministry of Health MCH Coordinators - Monitor maternal-health outcomes and coordinate targeted interventions

- Community Health Promoters (CHPs) - Conduct community and household follow-up and potentially prioritize women for visits

- Ministry of Health - Use evidence to support maternal-health planning and resource allocation

- Healthcare facilities - Strengthen continuity between ANC, delivery and PNC

- NGOs and development partners - Target maternal-health programmes and resources

- Researchers and public-health analysts - Use evidence to understand factors associated with PNC utilization

- Women and newborns - Ultimate beneficiaries of improved continuity of maternal and newborn care

## 1.5 Metrics of Success

The success of the project will be evaluated at both technical and practical levels.

**Model Performance**

- The primary metric will be PR-AUC, because the project focuses on identifying the minority group of women who miss timely postnatal care.

**Additional metrics will include:**

- Recall — ability to identify women who miss timely PNC
- Precision — proportion of women identified as high risk who actually miss timely PNC
- F1-score — balance between precision and recall
- ROC-AUC — overall discrimination
- Confusion matrix — assessment of classification errors

**Explainability**

- The model should be able to provide meaningful explanations for individual predictions using SHAP, showing the major factors that increase or decrease predicted risk.

**Practical Success**

The final prototype will be considered successful if a user can:

Enter a woman's profile → receive a predicted probability of missing timely PNC → see the risk classification → understand the main factors contributing to the prediction.

## 2. Data Understanding

In [2]:
# Setup: extract source files
import os
import zipfile
import pandas as pd
import numpy as np
import geopandas as gpd

RAW_DIR = "../data/raw"

zip_paths = {
    "recode": os.path.join(RAW_DIR, "KENR8CDT.zip"),
    "geo": os.path.join(RAW_DIR, "KEGE8AFL.zip")
}

for name, zpath in zip_paths.items():
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(RAW_DIR)

In [3]:
# Load pregnancy & postnatal care recode
# convert_categoricals=False avoids a ValueError caused by duplicate value labels on some
# variables (e.g. v326); raw numeric codes are retained and decoded selectively as needed.
dta_path = os.path.join(RAW_DIR, "KENR8CDT", "KENR8CFL.DTA")

reader = pd.io.stata.StataReader(dta_path)
df = reader.read(convert_categoricals=False)
variable_labels = reader.variable_labels()
value_labels = reader.value_labels()

print(f"Loaded: {df.shape[0]:,} records x {df.shape[1]} variables")

Loaded: 13,184 records x 919 variables


In [4]:
# Load GPS cluster file (used later only for mapping, not as a model input)
geo_path = os.path.join(RAW_DIR, "KEGE8AFL", "KEGE8AFL.shp")
gdf = gpd.read_file(geo_path)

print(f"Loaded: {gdf.shape[0]:,} survey clusters")

Loaded: 1,691 survey clusters


## Eligible Population

This recode is **pregnancy-indexed**, not birth-indexed. The eligibility variable is `m80`
("pregnancy outcome for this section"), which flags each woman's most recent pregnancy outcome:

| Code | Meaning |
|---|---|
| 1 | Most recent live birth |
| 3 | Most recent stillbirth |

Filtering to `m80 ∈ {1, 3}` restricts the data to each woman's most recent live birth or stillbirth —
the population this project defines as eligible for postnatal-care risk stratification. Stillbirths
are retained by design: mothers who experience a stillbirth still require postpartum care.

In [5]:
eligible = df[df["m80"].isin([1, 3])].copy()

print(f"Eligible records: {eligible.shape[0]:,}")
print(f"Unique women: {eligible['caseid'].nunique():,}")
print(f"Unique clusters: {eligible['v001'].nunique():,}")

Eligible records: 10,606
Unique women: 10,540
Unique clusters: 1,680


## Target Variable: Missed Timely Postnatal Care

The mother's postnatal check is captured across two DHS routing paths:
- **Pre-discharge check** (facility path): `m62` (checked, yes/no) → `m63` (timing)
- **Post-discharge / home-delivery check** (catch-all path): `m66` (checked, yes/no) → `m67` (timing)

Timing codes follow the DHS convention: `1XX` = hours, `2XX` = days, `3XX` = weeks, `998` = don't know.

**Target rule** (any provider — matches the standard global PNC-timeliness indicator, which does not
restrict by provider type):
- **0 (timely):** a check occurred within 48 hours on either path (hours `100`–`148`, or days `201`–`202`)
- **1 (missed):** no check occurred on either path, or the earliest check occurred at 3+ days / weeks
- **Excluded:** timing coded `998` (don't know) — only 40 records (0.38%), too small to justify an
  imputation assumption

In [6]:
def classify_check(check_flag, timing_code):
    if check_flag == 1:
        if pd.isna(timing_code) or timing_code == 998:
            return "unknown"
        elif 100 <= timing_code <= 148 or 201 <= timing_code <= 202:
            return "timely"
        elif timing_code >= 203:
            return "late"
        return "unknown"
    elif check_flag == 0:
        return "no_check"
    return "not_applicable"

def combine_target(pre, post):
    if pre == "timely" or post == "timely":
        return 0
    if pre == "unknown" or post == "unknown":
        return np.nan
    return 1

eligible["pre_discharge_status"] = eligible.apply(lambda r: classify_check(r["m62"], r["m63"]), axis=1)
eligible["post_discharge_status"] = eligible.apply(lambda r: classify_check(r["m66"], r["m67"]), axis=1)
eligible["missed_timely_pnc"] = eligible.apply(
    lambda r: combine_target(r["pre_discharge_status"], r["post_discharge_status"]), axis=1
)

model_df = eligible[eligible["missed_timely_pnc"].notna()].copy()
model_df["missed_timely_pnc"] = model_df["missed_timely_pnc"].astype(int)

print(f"Final modelling population: {model_df.shape[0]:,} records")
print(model_df["missed_timely_pnc"].value_counts(normalize=True).mul(100).round(1))

Final modelling population: 10,570 records
missed_timely_pnc
0    74.3
1    25.7
Name: proportion, dtype: float64


**Validation:** 74.3% timely / 25.7% missed. A stricter "quality PNC" indicator (all components,
not just any check) from independent KDHS 2022 analyses sits at 32.7–39% — consistent with our
basic timeliness measure being meaningfully higher, since it only requires one qualifying check.

## Candidate Predictors and Missingness

Predictors were mapped from the project proposal to their DHS variable codes, then checked for
missingness within the eligible population.

In [7]:
predictor_vars = {
    "v012": "Maternal age", "v106": "Education level", "v501": "Marital status",
    "v714": "Employment status", "v190": "Household wealth", "v201": "Children ever born",
    "m10": "Pregnancy intention", "m14": "Number of ANC visits", "m13": "Timing of first ANC visit",
    "m15": "Place of delivery", "m17": "Caesarean delivery", "v025": "Urban/rural residence",
    "v024": "Region"
}

summary = []
for var, label in predictor_vars.items():
    n_missing = eligible[var].isna().sum()
    summary.append({
        "Variable": var, "Description": label,
        "Missing": n_missing, "Missing %": round(100 * n_missing / len(eligible), 1)
    })

pd.DataFrame(summary)

,Variable,Description,Missing,Missing %
0,v012,Maternal age,0,0.0
1,v106,Education level,0,0.0
2,v501,Marital status,0,0.0
3,v714,Employment status,0,0.0
4,v190,Household wealth,0,0.0
5,v201,Children ever born,0,0.0
6,m10,Pregnancy intention,0,0.0
7,m14,Number of ANC visits,0,0.0
8,m13,Timing of first ANC visit,399,3.8
9,m15,Place of delivery,0,0.0


## Key Findings Summary

1. **Eligible population confirmed:** 10,606 pregnancy records (10,540 women, 1,680 clusters),
   matching the proposal's expected ~10,603.
2. **Target validated:** 10,570 records with a usable target; 74.3% timely / 25.7% missed timely PNC.
3. **Most core predictors are fully populated** (0% missing): age, education, marital status,
   employment, wealth, parity, pregnancy intention, ANC visit count, delivery place, caesarean status,
   residence, region.
4. **Structural gap identified:** healthcare-access-barrier variables (permission/money/distance/going
   alone) were only asked to half the sample (DHS long/short questionnaire split) — excluded from the
   main predictor set, noted as a limitation.
5. **33 unused "country-specific" template columns** (100% missing) identified for removal during
   data preparation.

## Structural Issues Identified (Handed Off to Data Cleaning)

Two structural, non-random data issues were identified during this understanding phase.
They are flagged here for visibility, with the actual fix left to the Data Cleaning stage:

1. **Long/short questionnaire split.** Four healthcare-access-barrier variables
   (`v467b`, `v467c`, `v467d`, `v467f` — permission, money, distance, going alone) are missing for
   exactly 47.4% of eligible records. This is not random or poor-quality data — it aligns perfectly
   with `sshort` (long/short questionnaire flag): these items were simply never asked of the
   short-questionnaire respondents. These variables are conceptually relevant to the project's goal
   of explaining access barriers, but the missingness is structural rather than fixable through
   imputation.

2. **Unused "country-specific" template columns.** 33 columns across the ANC-provider (`m2`),
   delivery-assistance (`m3`), and ANC-location (`m57`) variable groups are 100% missing — these are
   DHS template placeholders for country-specific categories that Kenya's 2022 questionnaire did not
   use, not a data-quality problem.

**Recommendation for Data Cleaning:** drop the 33 placeholder columns outright; decide whether to
exclude the four `v467` variables from the main model or scope them into a separate sub-analysis on
the long-questionnaire subsample only.

## 3. Data Preparation

*(Construct Data / Select Data — build `missed_timely_pnc` from `m62/m63` + `m66/m67`, select the ANC/delivery/demographic feature groups, drop the `m64`, `m68–m81` leakage series.)*

In [ ]:
# TODO: target construction + feature selection

## 4. Modeling

*(XGBoost baseline + tuning.)*

In [ ]:
# TODO: train/test split + XGBoost model

## 5. Evaluation

*(Metrics + SHAP explainability.)*

In [ ]:
# TODO: evaluation metrics + SHAP plots

## 6. Deployment

*(Streamlit app — see `/app`.)*

In [ ]:
# TODO: link/summary of Streamlit deployment